In [14]:
import json
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
from roboflow import Roboflow
from dotenv import load_dotenv

First time downloading the annotations (if annotation are already downloaded can be ignored)

In [15]:
# securely load API Key, necessary to download the dataset
# load_dotenv()
# rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))

# project = rf.workspace("bachelorthesis").project("8-ball-pool-l530o")
# dataset = project.version(3).download("coco")

In [16]:
def get_image_metadata(target_filename:str, annotation_path:str = './data/_annotations.coco.json'):
    # loop but there only is a train split
    splits = ["train"]
    
    for split in splits:        
        # skip if the folder/file doesn't exist
        if not os.path.exists(annotation_path):
            continue
            
        with open(annotation_path, 'r') as f:
            coco_data = json.load(f)
        
        category_map = {cat['id']: cat['name'] for cat in coco_data['categories']}

        # search for the image in this split
        image_id = None
        for img in coco_data['images']:
            if img['file_name'].startswith(target_filename) or img.get('extra', {}).get('name') == target_filename:
                image_id = img['id']
                break
        
        # found in this split, process and return
        if image_id is not None:
            image_annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == image_id and category_map[ann['category_id']] in ['Striped', 'Solid', 'Black', 'Cue']]
            
            return {
                "filename": target_filename,
                "split_found_in": split,
                "total_balls": len(image_annotations),
                "ball_list": [category_map[ann['category_id']] for ann in image_annotations],
                "raw_bboxes": [ann['bbox'] for ann in image_annotations],
                "numbers": [ann['number'] for ann in image_annotations]
            }

    # if it loops through all splits and finds nothing (should not unless)
    return f"Metadata for {target_filename} not found in train, valid, or test sets."

In [17]:
# 1 image example
prof_image = "3f_png" 
metadata = get_image_metadata(target_filename=prof_image)
print(metadata)

{'filename': '3f_png', 'split_found_in': 'train', 'total_balls': 15, 'ball_list': ['Striped', 'Striped', 'Striped', 'Striped', 'Striped', 'Striped', 'Solid', 'Solid', 'Solid', 'Solid', 'Solid', 'Solid', 'Solid', 'Cue', 'Black'], 'raw_bboxes': [[1337, 540, 62.68, 65.53], [1115, 676, 66.67, 72.36], [1028, 681, 64.39, 70.65], [1709, 488, 62.68, 66.1], [2372, 1112, 77.49, 82.05], [1871, 1268, 79.77, 84.9], [1964, 1353, 83.19, 82.05], [1220, 660, 64.96, 67.81], [963, 662, 68.38, 66.67], [931, 1127, 79.2, 80.91], [1963, 748, 65.5, 69.82], [2150, 535, 59.03, 67.66], [2347, 559, 64.06, 64.78], [2334, 636, 64.06, 66.94], [1697, 1211, 77.02, 79.18]], 'numbers': [13, 11, 12, 9, 15, 14, 7, 3, 6, 2, 1, 4, 5, 0, 8]}


In [18]:
def extract_table_contour(hsv_image, img_area):
    """
    Tries multiple HSV thresholds and validate the contour.
    """
    # threshold tiers
    # strict
    strict_lower = np.array([100, 150, 0])
    strict_upper = np.array([140, 255, 255])
    
    # loose
    loose_lower = np.array([95, 60, 20]) 
    loose_upper = np.array([145, 255, 255])
    
    thresholds_to_try = [
        ("Strict", strict_lower, strict_upper),
        ("Loose", loose_lower, loose_upper)
    ]
    
    # requirements to be considered a table
    MIN_AREA_RATIO = 0.125 # table must be at least 12.5% of the image
    MIN_EXTENT = 0.40     # table must fill at least 40% of its bounding box
    
    for name, lower, upper in thresholds_to_try:
        table_mask = cv2.inRange(hsv_image, lower, upper)
        contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            continue
            
        # get the largest blue contour should be the table
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        
        # --- VALIDATION CHECKS ---
        # check size
        if area < (img_area * MIN_AREA_RATIO):
            print(f"[{name} Threshold] Failed: Largest contour too small ({area} vs {img_area * MIN_AREA_RATIO})")
            continue
            
        # check shape (Extent)
        x, y, w, h = cv2.boundingRect(largest_contour)
        rect_area = w * h
        extent = float(area) / rect_area
        
        if extent < MIN_EXTENT:
            print(f"[{name} Threshold] Failed: Extent too low ({extent:.2f}). Not rectangular enough.")
            continue
            
        print(f"[{name} Threshold] Success! Table found.")


        # clean mask where only the table will be
        surface_mask = np.zeros_like(table_mask)
        cv2.drawContours(surface_mask, [largest_contour], -1, 255, -1)

        return surface_mask, table_mask
        
    # If all loops fail
    return None, None

In [19]:
def detect_balls(img_path:str, show_plots:bool=False):
  
  img = cv2.imread(img_path)
  height, width = img.shape[:2]
  img = cv2.resize(img, (800, int(800 * height / width)))

  img_area = img.shape[0] * img.shape[1]

    
  #TABLE MASK
  hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
  surface_mask, table_mask = extract_table_contour(hsv_image=hsv, img_area=img_area)

  img_test = img.copy()
  gray = cv2.cvtColor(img_test, cv2.COLOR_BGR2GRAY)
    
  gray = cv2.bitwise_and(gray, gray, mask=table_mask)
  kernel = np.ones((15, 15), np.uint8)
  eroded_mask = cv2.erode(surface_mask, kernel, iterations=1)
  masked_frame = cv2.bitwise_and(gray, gray, mask=eroded_mask)
    
  # Blur first — reduces noise, improves circle detection significantly
  blurred = cv2.GaussianBlur(masked_frame, (3, 3), 0)
    
  circles = cv2.HoughCircles(
      blurred,
      cv2.HOUGH_GRADIENT_ALT,
      dp=1.5,
      minDist=10,
      param1=100,
      param2=0.5,
      minRadius=4,
      maxRadius=20
  )
    


  final_circles = []
  if circles is not None:
    median_test_radius = np.median(circles[0, :, 2])
    if show_plots:
      print(f"Median radius: {median_test_radius}")

      
    circles = np.round(circles[0]).astype(int)
    for x, y, r in circles:
      if show_plots:
        print(f"Circle found at ({x}, {y}) with radius {r}")
        
      # draw the outer circle
      cv2.circle(img_test, (x, y), r, (0, 255, 0), 2)
      # draw the center of the circle
      cv2.circle(img_test, (x, y), 2, (0, 0, 255), 3)
      final_circles.append([x, y, r])



  if show_plots:
  
    f, axarr = plt.subplots(1, 4, figsize=(20, 10))
    axarr[0].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB))
    axarr[0].set_title("Blurred Image")
    axarr[0].axis("off")

    axarr[1].imshow(cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB))
    axarr[1].set_title("Detected Circles")
    axarr[1].axis("off")

    axarr[2].imshow(surface_mask, cmap="gray")
    axarr[2].set_title("Surface Mask")
    axarr[2].axis("off")

    axarr[3].imshow(eroded_mask, cmap="gray")
    axarr[3].set_title("Eroded Surface Mask")
    axarr[3].axis("off")

  return final_circles

In [20]:
def get_color_name(hue, sat, val):
    """Maps an OpenCV HSV value to a pool ball color."""
    # first handle Black and White (since they rely entirely on saturation and value not hue)
    if val < 60:
        return "Black"
    if sat < 60 and val > 120:
        return "White"
        
    # pool ball hues in OpenCV scale (0-179)
    if (hue >= 0 and hue < 12) or (hue > 165):
        return "Red"        # 3 or 11
    elif hue >= 12 and hue < 25:
        # orange and brown share similar hues. brown is just dark orange.
        if val < 150: return "Brown" # 7 or 15
        else: return "Orange"        # 5 or 13
    elif hue >= 25 and hue < 40:
        return "Yellow"     # 1 or 9
    elif hue >= 40 and hue < 85:
        return "Green"      # 6 or 14
    elif hue >= 85 and hue < 130:
        return "Blue"       # 2 or 10
    elif hue >= 130 and hue <= 165:
        return "Purple"     # 4 or 12
    
    return "Unknown"

In [21]:
def analyze_ball_color(img, cx, cy, r, show_plot=False):
    """
    Crops the ball, masks it, analyzes HSV colors, and plots a histogram.
    """
    height, width = img.shape[:2]
    
    # convert circle to bounding box (with safety checks so we don't go off-image)
    x_min = max(0, cx - r)
    y_min = max(0, cy - r)
    x_max = min(width, cx + r)
    y_max = min(height, cy + r)
    
    # crop the Bounding Box
    roi_bgr = img[y_min:y_max, x_min:x_max]
    
    # create a circular mask to ignore the table corners in the bounding box
    mask = np.zeros(roi_bgr.shape[:2], dtype=np.uint8)
    # the center of the circle in the cropped ROI is at (r, r) if it didn't hit an edge
    local_cx = cx - x_min
    local_cy = cy - y_min
    cv2.circle(mask, (local_cx, local_cy), r, 255, -1)
    
    # convert to HSV
    roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    
    # Extract only the pixels inside our circular mask
    hsv_pixels = roi_hsv[mask == 255]
    
    if len(hsv_pixels) == 0:
        return "Unknown", "Unknown"
        
    hues = hsv_pixels[:, 0]
    sats = hsv_pixels[:, 1]
    vals = hsv_pixels[:, 2]
    
    # count White, Black, and Colored pixels
    # threshold might need adjustment
    white_mask = (sats < 60) & (vals > 120)
    black_mask = (vals < 60)
    
    white_count = np.sum(white_mask)
    black_count = np.sum(black_mask)
    total_pixels = len(hsv_pixels)
    
    # rest are 'colored' pixels
    color_mask = ~(white_mask | black_mask)
    color_hues = hues[color_mask]
    color_count = len(color_hues)
    
    # calculate the dominant Hue
    dominant_color = "Unknown"
    if color_count > 0:
        # calculate histogram of the colored pixels (0-180 for OpenCV Hue range)
        hist, bins = np.histogram(color_hues, bins=18, range=(0, 180))
        # find the bin with the most pixels
        peak_bin = np.argmax(hist)
        dominant_hue = (bins[peak_bin] + bins[peak_bin+1]) / 2
        
        # get a representative sat and val (medians of the colored pixels)
        med_sat = np.median(sats[color_mask])
        med_val = np.median(vals[color_mask])
        
        dominant_color = get_color_name(dominant_hue, med_sat, med_val)
    
    # solid vs striped vs cue vs black depends on the ration of white/black that exists in the ball
    white_ratio = white_count / total_pixels
    black_ratio = black_count / total_pixels
    
    if white_ratio > 0.85:
        ball_type = "Cue"
        final_color = "White"
    elif black_ratio > 0.60:
        ball_type = "Solid"
        final_color = "Black"
    # ff it has a significant amount of white and also a distinct color
    elif white_ratio > 0.20 and color_count > 0: 
        ball_type = "Striped"
        final_color = dominant_color
    # otherwise, it's mostly color
    else:
        ball_type = "Solid"
        final_color = dominant_color

    if show_plot:
        fig, axs = plt.subplots(1, 3, figsize=(15, 4))
        
        # raw BGR crop
        axs[0].imshow(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB))
        axs[0].set_title(f"Crop: {ball_type} {final_color}")
        axs[0].axis('off')
        
        # mask isolated (the ball itself)
        masked_bgr = cv2.bitwise_and(roi_bgr, roi_bgr, mask=mask)
        axs[1].imshow(cv2.cvtColor(masked_bgr, cv2.COLOR_BGR2RGB))
        axs[1].set_title("Masked Ball")
        axs[1].axis('off')
        
        # Hue histogram (only for colored pixels)
        if color_count > 0:
            axs[2].hist(color_hues, bins=36, range=(0, 180), color='purple', alpha=0.7)
            axs[2].set_title("Hue Distribution (Only Color Pixels)")
            axs[2].set_xlabel("Hue (0-179)")
            axs[2].set_ylabel("Pixel Count")
        else:
            axs[2].text(0.5, 0.5, "No distinct color (Cue/Black)", ha='center')
            axs[2].axis('off')
            
        plt.tight_layout()
        plt.show()
        
    return ball_type, final_color

In [22]:
def detect_number (img_path, circles, show_plot=False):
    
    img = cv2.imread(img_path)

    height, width = img.shape[:2]
    img = cv2.resize(img, (800, int(800 * height / width)))

    final_number = []
    for cx, cy, r in circles:
        ball_type, color = analyze_ball_color(img, cx, cy, r, show_plot)

        if ball_type == 'Cue':
            final_number.append(0)
        elif ball_type == 'Black':
            final_number.append(8)
        else:
            num_off_set = 0
            if ball_type == 'Striped':
                num_off_set = 8

            match color:
                case "Yellow":
                    final_number.append(1 + num_off_set)

                case "Blue":
                    final_number.append(2 + num_off_set)

                case "Red":
                    final_number.append(3 + num_off_set)

                case "Purple":
                    final_number.append(4 + num_off_set)

                case "Orange":
                    final_number.append(5 + num_off_set)

                case "Green":
                    final_number.append(6 + num_off_set)

                case "Brown":
                    final_number.append(7 + num_off_set)

                case _:
                    print("NO NUMBER DETECTED")
                    final_number.append(-1)
                
    return final_number

In [23]:
def calculate_iou(boxA, boxB):
    # boxes folow the format: [x_min, y_min, x_max, y_max]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # compute intersection area
    interArea = max(0, xB - xA) * max(0, yB - yA)

    # compute areas of both bounding boxes
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # calculate Intersection over Union
    if float(boxAArea + boxBArea - interArea) == 0:
        return 0.0
    
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def evaluate_image_detections(truth_boxes_coco, pred_circles, true_number_coco, pred_numbers, scale_x, scale_y, iou_threshold=0.5):
    """
    truth_boxes_coco: list of [x, y, width, height] from Roboflow
    pred_circles: list of [x_center, y_center, radius] from HoughCircles
    scale_x, scale_y: scaling factors to adjust ground truth to the resized 800px image
    """
    
    # convert COCO truth boxes to [x_min, y_min, x_max, y_max] apply scaling due to image resizing that happens in detect_balls()
    t_boxes = []
    for x, y, w, h in truth_boxes_coco:
        x_min = x * scale_x
        y_min = y * scale_y
        x_max = (x + w) * scale_x
        y_max = (y + h) * scale_y
        t_boxes.append([x_min, y_min, x_max, y_max])

    # convert predicted circles to [x_min, y_min, x_max, y_max]
    p_boxes = []
    for cx, cy, r in pred_circles:
        p_boxes.append([cx - r, cy - r, cx + r, cy + r])

    matched_ious = []
    detected_truth_indices = set()

    # will count the number of correctly predicted balls
    correct_number_count = 0
    incorrect_number_count = 0
    # match predictions to truth boxes
    for i, p_box in enumerate(p_boxes):
        best_iou = 0
        best_t_idx = -1
        
        for t_idx, t_box in enumerate(t_boxes):

            # if that ball was already claimed by another pred circle then it can't be claimed by this pred circle
            if t_idx in detected_truth_indices:
                continue

            iou = calculate_iou(p_box, t_box)
            if iou > best_iou:
                best_iou = iou
                best_t_idx = t_idx


        # just accept balls above the threshold
        if best_iou >= iou_threshold:
            matched_ious.append(best_iou)
            detected_truth_indices.add(best_t_idx)

            # best_t_idx is the index of the true bbox that was best matched to the curent pred bbox being analized
            if (true_number_coco[best_t_idx] == pred_numbers[i]):
                correct_number_count+=1
            else:
                # necessary because there might be balls predicted that are not balls and as such can't count for the incorrect classified
                incorrect_number_count+=1



    true_count = len(truth_boxes_coco)
    pred_count = len(pred_circles)
    undetected_count = true_count - len(detected_truth_indices)

    return {
        "true_count": true_count,
        "pred_count": pred_count,
        "correct_number_count": correct_number_count,
        "incorrect_number_count": incorrect_number_count,
        "undetected_count": undetected_count,
        "matched_ious": matched_ious
    }

In [24]:
TEST_DATA_PATH = "./data/development_set/"
ims_path = os.listdir(TEST_DATA_PATH)

total_images = 0
total_absolute_error = 0
exact_count_matches = 0
total_truth_balls = 0
total_undetected_balls = 0
total_correct_number = 0
total_incorrect_number = 0
all_ious = []

for path in ims_path:
        
    im_name = re.findall(pattern="[0-9]+._png", string=path)[0]
    img_full_path = os.path.join(TEST_DATA_PATH, path)
    
    # get metadata/ground truth
    metadata = get_image_metadata(target_filename=im_name)
    if isinstance(metadata, str): 
        print(metadata) # Metadata not found
        continue
        
    # original image dimensions to calculate scaling factors
    original_img = cv2.imread(img_full_path)

    #error reading image
    if original_img is None:
        continue
    orig_height, orig_width = original_img.shape[:2]
    
    # dimensions  detect_balls function resizes to:
    resized_width = 800
    resized_height = int(800 * orig_height / orig_width)
    
    scale_x = resized_width / orig_width
    scale_y = resized_height / orig_height

    #-----------------DETECTION CALL-----------------
    circles = detect_balls(img_path=img_full_path, show_plots=False)

    numbers = detect_number(img_path=img_full_path, circles=circles, show_plot=False)
    #-----------------DETECTION CALL-----------------

    
    #-----------------Evaluation CALL-----------------
    stats = evaluate_image_detections(
        truth_boxes_coco=metadata["raw_bboxes"],
        pred_circles=circles,
        true_number_coco= metadata["numbers"],
        pred_numbers = numbers,
        scale_x=scale_x, 
        scale_y=scale_y,
        iou_threshold=0.5 # could be tunned ask professor
    )
    #-----------------Evaluation CALL-----------------

    total_images += 1
    total_absolute_error += abs(stats["pred_count"] - stats["true_count"])
    
    if stats["pred_count"] == stats["true_count"]:
        exact_count_matches += 1
        
    total_truth_balls += stats["true_count"]
    total_undetected_balls += stats["undetected_count"]
    total_correct_number += stats["correct_number_count"]
    total_incorrect_number += stats["incorrect_number_count"]
    all_ious.extend(stats["matched_ious"])
    
    print(f"Processed {im_name} | True: {stats['true_count']} | Pred: {stats['pred_count']} | Undetected: {stats['undetected_count']}")




#-----------------METRICS CALCULATION-----------------
print("\n" + "="*30)
print("FINAL PIPELINE METRICS")
print("="*30)

# Ball Count MAE
mae = total_absolute_error / total_images if total_images > 0 else 0
print(f"Ball Count MAE: {mae:.2f} balls")

# Ball Count Accuracy
accuracy = (exact_count_matches / total_images) * 100 if total_images > 0 else 0
print(f"Ball Count Accuracy (Exact Match): {accuracy:.2f}%")

# BBox Detection IOU
mean_iou = np.mean(all_ious) if all_ious else 0
print(f"Average BBox IOU (True Positives): {mean_iou:.4f}")

# Average % of undetected balls (False Negatives)
undetected_pct = (total_undetected_balls / total_truth_balls) * 100 if total_truth_balls > 0 else 0
print(f"Percentage of Undetected Balls: {undetected_pct:.2f}%")

# Average % of correct numbered balls
correct_number_pct = (total_correct_number / (total_correct_number+total_incorrect_number)) * 100 if (total_correct_number+total_incorrect_number) > 0 else 0
print(f"Percentage of correct numbers indentified in Balls: {correct_number_pct:.2f}%")

# Average % of incorrect numbered balls
incorrect_number_pct = (total_incorrect_number / (total_correct_number+total_incorrect_number)) * 100 if (total_correct_number+total_incorrect_number) > 0 else 0
print(f"Percentage of incorrect numbers indentified in Balls: {incorrect_number_pct:.2f}%")

[Strict Threshold] Success! Table found.
Processed 21a_png | True: 14 | Pred: 8 | Undetected: 9
[Strict Threshold] Success! Table found.
Processed 117_png | True: 9 | Pred: 8 | Undetected: 2
[Strict Threshold] Success! Table found.
Processed 7a_png | True: 14 | Pred: 12 | Undetected: 3
[Strict Threshold] Success! Table found.
Processed 127_png | True: 16 | Pred: 11 | Undetected: 10
[Strict Threshold] Success! Table found.
Processed 32_png | True: 5 | Pred: 3 | Undetected: 3
[Strict Threshold] Success! Table found.
Processed 132_png | True: 11 | Pred: 9 | Undetected: 2
[Strict Threshold] Success! Table found.
Processed 165_png | True: 12 | Pred: 9 | Undetected: 3
[Strict Threshold] Success! Table found.
Processed 156_png | True: 5 | Pred: 5 | Undetected: 0
[Strict Threshold] Success! Table found.
Processed 91_png | True: 15 | Pred: 10 | Undetected: 6
[Strict Threshold] Failed: Largest contour too small (8369.0 vs 50000.0)
[Loose Threshold] Success! Table found.
Processed 16a_png | True: